# Gemma 4 Classroom Adaptation\n\n## One Science Lesson, Three Reading Levels, Zero Cloud Dependence\n\nFine-tuning Gemma 4 with Unsloth to help middle school science teachers serve mixed reading levels in low-bandwidth classrooms.

## Problem\n\nA middle school science teacher may need to serve students with very different reading levels in the same room. Rewriting the same lesson by hand for multiple learners takes time teachers do not have, and cloud-first AI is not always practical in low-connectivity school settings.\n\nThis project focuses on one narrow, high-value workflow: take one science lesson and adapt it into `below`, `on`, and `above` reading-level versions while preserving the underlying science facts.

## Approach\n\n- Base model: `google/gemma-4-E4B-it`\n- Fine-tuning method: Unsloth LoRA\n- Task: single-target rewriting with target labels `below`, `on`, and `above`\n- Constraint: preserve a list of must-keep science facts\n- Output format: `Adapted Lesson` plus `Key Concepts Preserved`\n\nWe use a held-out evaluation set of middle school science passages and compare base Gemma 4 against the fine-tuned adapter on fact coverage, level control, and teacher usefulness.

In [ ]:
from pathlib import Path\nimport json\n\neval_path = Path('../artifacts/evals/first_run_eval.json')\nif eval_path.exists():\n    report = json.loads(eval_path.read_text())\n    summary = {\n        'base': report['base']['summary'],\n        'tuned': report['tuned']['summary'],\n    }\n    summary\nelse:\n    print(f'Missing eval artifact: {eval_path.resolve()}')

## Benchmark Results\n\nTo test whether post-training improved the exact classroom workflow we cared about, we compared **base Gemma 4** with our **Unsloth fine-tuned Gemma 4 adapter** on a held-out set of middle school science passages. Each example asked the model to adapt one lesson for a target reading level: `below`, `on`, or `above`.\n\nWe evaluated three criteria:\n\n- **Fact coverage:** whether the adapted lesson preserved the required science facts\n- **Reading-level alignment:** whether the lesson matched the requested complexity band\n- **Teacher usefulness:** a classroom proxy combining factual preservation, level control, and output completeness

| Metric | Base Gemma 4 | Tuned Gemma 4 | Delta |\n|---|---:|---:|---:|\n| Avg fact coverage | 0.417 | 1.000 | +0.583 |\n| Avg teacher usefulness | 0.408 | 0.967 | +0.559 |\n| Within target band rate | 0.417 | 0.833 | +0.416 |

These results matter because the project is intentionally narrow. We are not benchmarking general educational QA or open-ended tutoring. We are measuring one transformation: can the model reliably turn a teacher's science lesson into a level-appropriate classroom version without changing the science?

## Qualitative Example\n\nOne of the clearest held-out examples was **`electric_circuits_001` at the `on` level**. Under the same prompt, the base model failed to produce a usable adapted lesson. The tuned model returned a complete structured response, preserved all five required circuit facts, and matched the requested classroom format.\n\nWe saw the same pattern on other held-out `on`-level prompts, including **`cells_001`** and **`atoms_molecules_001`**. In each case, the tuned model was more reliable at producing a complete classroom-ready output while preserving the science content.

In [ ]:
from pathlib import Path\nimport json\n\neval_path = Path('../artifacts/evals/first_run_eval.json')\nif eval_path.exists():\n    report = json.loads(eval_path.read_text())\n    for base_row, tuned_row, base_ex, tuned_ex in zip(report['base']['rows'], report['tuned']['rows'], report['base_examples'], report['tuned_examples']):\n        if base_row['source_id'] == 'electric_circuits_001' and base_row['target_level'] == 'on':\n            print('Base summary:', base_row)\n            print()\n            print('Tuned summary:', tuned_row)\n            print()\n            print('Base raw output:\n')\n            print(base_ex['generated_text'])\n            print('\n' + '=' * 80 + '\n')\n            print('Tuned raw output:\n')\n            print(tuned_ex['generated_text'])\n            break\nelse:\n    print(f'Missing eval artifact: {eval_path.resolve()}')

## Why This Supports The Project Story\n\nThe core story of this project is that one teacher may need to serve several reading levels from the same lesson, often with limited time and unreliable connectivity. The benchmark supports that story directly. The tuned model is not simply different from the base model. It is **more dependable on the exact adaptation task the teacher needs**.\n\nThat is the technical claim behind the demo: one science lesson can become multiple level-appropriate versions quickly, and the tuned Gemma 4 model does a better job preserving facts, following the expected structure, and producing something a teacher could actually use.